# 실제 모델 NumPy 재계산 — 독립 경로 대조

TypeScript 구현이 규칙을 제대로 옮겼는지 확인하기 위해, **파이썬으로 모델을 처음부터
다시 구현**해 결과를 맞춰 본다. 두 구현은 코드를 공유하지 않으므로, 전이행렬이 서로
같다면 전사·논리 오류가 없다는 강한 증거가 된다. (계획서 §7 · §11.4)

먼저 TypeScript 쪽 결과를 내보내 둔다.

```bash
npm run export:model
```

## 1. 파이썬으로 모델을 다시 구현한다

$\pi$ 는 금액에 의존하지 않으므로 칸의 **종류**만 있으면 된다.
황금열쇠 6칸, 무인도, 우주여행, 그리고 이동 카드 12장이 전부다.

In [1]:
import numpy as np
from collections import deque

BOARD = 40
GK = {2, 7, 12, 17, 22, 34}
ISLAND, SPACE = 10, 30

def wrap(x):
    return x % BOARD

def passes_start(a, b):
    """a 에서 b 로 앞으로 갈 때 출발선을 지나는 횟수."""
    steps = wrap(b - a)
    return 1 if steps > 0 and a + steps >= BOARD else 0

# 위치를 바꾸는 카드 11장 + 세계일주(제자리) 1장 + 비이동 18장 = 30장.
# ('kind', 인자..., 장수)
CARDS = [
    ("via",   15,  1, 1),   # 항공여행    — 콩코드 경유 → 타이베이
    ("via",   28,  3, 1),   # 유람선 여행 — 퀸 엘리자베스 경유 → 베이징
    ("via",   32, 30, 1),   # 우주여행 초대권 — 컬럼비아호 경유 → 우주여행
    ("to",     0,     1),   # 고속도로
    ("to",    39,     1),   # 관광여행 · 서울
    ("to",    25,     1),   # 관광여행 · 부산
    ("to",     5,     1),   # 관광여행 · 제주도
    ("to",    20,     1),   # 사회복지기금 배당
    ("back",  -2,     1),   # 이사 · 뒤로 2칸
    ("back",  -3,     1),   # 이사 · 뒤로 3칸
    ("jail",           1),  # 무인도
    ("stay",          19),  # 세계일주 초대권 1 + 비이동 18
]
DECK = sum(c[-1] for c in CARDS)
assert DECK == 30

# 상태: 0..9 → 칸 0..9,  10..13 → J3 J2 J1 J0,  14..42 → 칸 11..39
LABELS = [f"c{i}" for i in range(10)] + ["J3", "J2", "J1", "J0"] + [f"c{i}" for i in range(11, 40)]
N = len(LABELS)
assert N == 43
IDX = {lab: i for i, lab in enumerate(LABELS)}

def cell_state(cell):
    """칸 번호 → 상태 인덱스. 10번은 J3(갓 갇힘)를 가리킨다."""
    return IDX["J3"] if cell == ISLAND else IDX[f"c{cell}"]

def dice_outcomes(faces=(1, 2, 3, 4, 5, 6)):
    out = {}
    p = 1 / len(faces) ** 2
    for a in faces:
        for b in faces:
            out[(a + b, a == b)] = out.get((a + b, a == b), 0) + p
    return [(s, d, pr) for (s, d), pr in out.items()]

def resolve(cell, depth=0):
    """착지 결과 목록: (확률, 위치, 턴종료). 위치는 칸 번호 또는 'JAIL'."""
    if cell == ISLAND:
        return [(1.0, "JAIL", True)]
    if cell == SPACE:
        return [(1.0, SPACE, True)]          # 더블과 무관하게 턴 종료
    if cell in GK and depth == 0:            # A6: 카드 이동 후에는 다시 뽑지 않는다
        out = []
        for card in CARDS:
            n = card[-1]
            p = n / DECK
            kind = card[0]
            if kind == "stay":
                out.append((p, cell, False))
            elif kind == "jail":
                out.append((p, "JAIL", True))
            else:
                target = wrap(cell + card[1]) if kind == "back" else card[2] if kind == "via" else card[1]
                # A11: 카드로 이동해도 더블이면 계속 굴린다 → 턴 종료는 목적지가 정한다
                out += [(p * q, pos, end) for q, pos, end in resolve(target, depth + 1)]
        return out
    return [(1.0, cell, False)]

def build_matrix():
    dice = dice_outcomes()
    P = np.zeros((N, N))

    def settle(row, pos, prob):
        row[IDX["J3"] if pos == "JAIL" else cell_state(pos)] += prob

    def normal_turn(row, start):
        """굴림을 레벨 단위로 집계한다 (더블 재굴림)."""
        active = np.zeros(BOARD)
        active[start] = 1.0
        for _ in range(64):
            carried = np.zeros(BOARD)
            for cell in range(BOARD):
                if active[cell] < 1e-15:
                    continue
                for total, is_double, dp in dice:
                    p = active[cell] * dp
                    for q, pos, ends in resolve(wrap(cell + total)):
                        m = p * q
                        if ends or not is_double:
                            settle(row, pos, m)
                        else:
                            carried[pos] += m
            if carried.sum() < 1e-15:
                return
            active = carried

    for i, label in enumerate(LABELS):
        row = P[i]
        if label == "c30":                        # 우주여행 — 주사위를 굴리지 않는다
            for target in range(BOARD):
                if target == SPACE:
                    continue
                for q, pos, _ in resolve(target):
                    settle(row, pos, q / (BOARD - 1))
        elif label in ("J3", "J2", "J1"):         # 무인도 대기 — 더블로만 탈출
            for total, is_double, dp in dice:
                if not is_double:
                    row[IDX[f"J{int(label[1]) - 1}"]] += dp
                    continue
                for q, pos, _ in resolve(wrap(ISLAND + total)):
                    settle(row, pos, dp * q)
        else:                                     # J0 포함 정상 턴
            normal_turn(row, ISLAND if label == "J0" else int(label[1:]))
        row /= row.sum()                          # 행 정규화 (§11.1)
    return P

P_py = build_matrix()
print("행 합이 모두 1:", np.allclose(P_py.sum(1), 1, atol=1e-12))
print("모든 확률 ≥ 0 :", (P_py >= 0).all())

행 합이 모두 1: True
모든 확률 ≥ 0 : True


## 2. TypeScript 결과를 읽어 대조한다

In [2]:
import json

with open("data/default.json") as f:
    ts = json.load(f)

P_ts = np.array(ts["matrix"])
pi_ts = np.array(ts["pi"])

print("상태 순서 일치 :", ts["labels"] == LABELS)
print("행렬 최대 차이 :", np.abs(P_ts - P_py).max())
print("두 전이행렬이 같다:", np.allclose(P_ts, P_py, atol=1e-12))

상태 순서 일치 : True
행렬 최대 차이 : 2.220446049250313e-16
두 전이행렬이 같다: True


## 3. 정상분포를 세 경로로 구한다

파이썬 고유벡터 · 파이썬 멱승법 · TypeScript 멱승법.

In [3]:
vals, vecs = np.linalg.eig(P_py.T)
k = int(np.argmin(np.abs(vals - 1)))
pi_eig = np.real(vecs[:, k]); pi_eig /= pi_eig.sum()

pi_pow = np.full(N, 1 / N)
for _ in range(3000):
    pi_pow = pi_pow @ P_py
    pi_pow /= pi_pow.sum()

print("파이썬 고유벡터 vs 멱승법 :", np.abs(pi_eig - pi_pow).sum())
print("파이썬 vs TypeScript      :", np.abs(pi_eig - pi_ts).sum())
print("확률벡터 조건 (합, 최솟값):", pi_ts.sum(), pi_ts.min())
print("잔차 ||πP − π||₁          :", np.abs(pi_ts @ P_py - pi_ts).sum())

파이썬 고유벡터 vs 멱승법 : 9.766493169749424e-16
파이썬 vs TypeScript      : 5.986530715595961e-15
확률벡터 조건 (합, 최솟값): 0.9999999999999997 0.013676245922370817
잔차 ||πP − π||₁          : 6.480926906249351e-15


## 4. 두 번째 고유값

TypeScript 는 $|\lambda_2|$ 를 두 경로로 구한다 — 디플레이션(이론)과 로그 회귀(실측).
여기서는 고유값을 **직접** 구해 어느 쪽이 맞는지 판정한다.

In [4]:
ev = np.linalg.eigvals(P_py)
order = np.argsort(-np.abs(ev))
print("절댓값이 큰 고유값 5개")
for i in order[:5]:
    print(f"  {ev[i]:+.8f}   |λ| = {abs(ev[i]):.8f}")

true_l2 = abs(ev[order[1]])
print()
print(f"참값 |λ₂|            = {true_l2:.8f}")
print(f"TS 디플레이션        = {ts['lambda2']['deflation']:.8f}  (진동: {ts['lambda2']['deflationOscillating']})")
print(f"TS 로그 회귀         = {ts['lambda2']['logRegression']:.8f}  (R² = {ts['lambda2']['logRegressionR2']:.6f})")

절댓값이 큰 고유값 5개
  +1.00000000+0.00000000j   |λ| = 1.00000000
  +0.22137214+0.66826373j   |λ| = 0.70397588
  +0.22137214-0.66826373j   |λ| = 0.70397588
  -0.37327094+0.47681516j   |λ| = 0.60554429
  -0.37327094-0.47681516j   |λ| = 0.60554429

참값 |λ₂|            = 0.70397588
TS 디플레이션        = 0.70370216  (진동: True)
TS 로그 회귀         = 0.70375277  (R² = 0.999974)


$\lambda_2$ 가 **복소 켤레쌍**이면 디플레이션의 스텝별 성장률이 진동한다.
계획서 §15가 예상한 상황이며, 그때는 로그 회귀 실측값을 주 결과로 쓰고
진동 자체를 복소 고유값의 증거로 보고한다.

## 5. 순수 순환 보드 — 해석적 정답 (§11.2)

특수 칸을 모두 끄면 $P$ 가 순환행렬이 되어 이중확률행렬이고, 따라서
$\pi_i = 1/40$ 이 **정확히** 성립한다. 손으로 증명 가능한 유일한 지점이다.

In [5]:
with open("data/plain-cyclic.json") as f:
    plain = json.load(f)

P0 = np.array(plain["matrix"])
pi0 = np.array(plain["pi"])
print("상태 수            :", len(pi0))
print("행 합 = 1          :", np.allclose(P0.sum(1), 1))
print("열 합 = 1 (이중확률):", np.allclose(P0.sum(0), 1))
print("π 와 1/40 의 최대 차:", np.abs(pi0 - 1 / 40).max())
print("한 턴 기대 굴림 수  :", plain["rolls"][0], " (이론 6/5 =", 6 / 5, ")")
print("턴당 출발선 통과    :", np.mean(plain["salaryPasses"]), " (이론 8.4/40 =", 8.4 / 40, ")")

상태 수            : 40
행 합 = 1          : True
열 합 = 1 (이중확률): True
π 와 1/40 의 최대 차: 2.983724378680108e-16
한 턴 기대 굴림 수  : 1.199999999999995  (이론 6/5 = 1.2 )
턴당 출발선 통과    : 0.20999999999999835  (이론 8.4/40 = 0.21000000000000002 )


## 6. 상위·하위 칸

$\pi$ 를 칸 단위로 모아 본다. 무인도 4상태는 10번 칸으로 합친다.

In [6]:
cells = np.array(ts["cells"])
by_cell = np.zeros(BOARD)
for i, c in enumerate(cells):
    by_cell[c] += pi_ts[i]

names = ["출발","타이베이","황금열쇠","베이징","마닐라","제주도","싱가포르","황금열쇠","카이로","이스탄불",
         "무인도","아테네","황금열쇠","코펜하겐","스톡홀름","콩코드","베른","황금열쇠","베를린","오타와",
         "수령처","부에노스아이레스","황금열쇠","상파울루","시드니","부산","하와이","리스본","퀸엘리자베스","마드리드",
         "우주여행","도쿄","컬럼비아호","파리","황금열쇠","로마","런던","뉴욕","기금접수처","서울"]

order = np.argsort(-by_cell)
print("합 =", by_cell.sum(), " 균등이면", 1 / 40)
print()
print("방문 확률 상위 5칸")
for i in order[:5]:
    print(f"  {i:2d} {names[i]:<10s} {by_cell[i]*100:6.3f}%   (균등 대비 {by_cell[i]*40:.2f}배)")
print()
print("방문 확률 하위 5칸")
for i in order[-5:]:
    print(f"  {i:2d} {names[i]:<10s} {by_cell[i]*100:6.3f}%   (균등 대비 {by_cell[i]*40:.2f}배)")

합 = 0.9999999999999998  균등이면 0.025

방문 확률 상위 5칸
  10 무인도        10.835%   (균등 대비 4.33배)
  30 우주여행        3.316%   (균등 대비 1.33배)
  20 수령처         3.039%   (균등 대비 1.22배)
  25 부산          2.814%   (균등 대비 1.13배)
   5 제주도         2.765%   (균등 대비 1.11배)

방문 확률 하위 5칸
  22 황금열쇠        1.590%   (균등 대비 0.64배)
   7 황금열쇠        1.473%   (균등 대비 0.59배)
  34 황금열쇠        1.470%   (균등 대비 0.59배)
   2 황금열쇠        1.388%   (균등 대비 0.56배)
  17 황금열쇠        1.368%   (균등 대비 0.55배)


## 결론

| 경로 | 확인 |
|---|---|
| 파이썬 독립 구현 vs TypeScript 전이행렬 | 위 셀 출력 참조 |
| 파이썬 고유벡터 vs 파이썬 멱승법 | 〃 |
| 파이썬 vs TypeScript 정상분포 | 〃 |
| 순수 순환 보드 = 1/40 (해석적 정답) | 〃 |
| $\|\lambda_2\|$ 참값 vs 로그 회귀 실측 | 〃 |

몬테카를로(§11.4의 두 번째 축)와 Kac 재귀시간(세 번째 축)은 Phase 6–7에서 붙인다.